In [0]:
def log_run(catalog, schema_name, run_context):
    """
    Logs comprehensive ETL run information to the logging table.
    """
    full_table_name = f"{catalog}.{schema_name}.etl_run_log"
    
    try:
        log_schema = StructType([
            StructField("run_id", StringType(), False),
            StructField("table_name", StringType(), False),
            StructField("log_datetime", TimestampType(), False),
            StructField("status", StringType(), False),
            StructField("load_type", StringType(), True),
            StructField("error_message", StringType(), True),
            StructField("duration_seconds", DoubleType(), True),
            StructField("duration_minutes", DoubleType(), True),
            StructField("read_duration_seconds", DoubleType(), True),
            StructField("write_duration_seconds", DoubleType(), True),
            StructField("source_row_count", LongType(), True),
            StructField("target_row_count", LongType(), True),
            StructField("inserted", IntegerType(), True),
            StructField("updated", IntegerType(), True),
            StructField("deleted", IntegerType(), True),
            StructField("copied", IntegerType(), True),
            StructField("incremental_interval", IntegerType(), True),
            StructField("was_forced_full_load", BooleanType(), True)
        ])

        def to_int(value):
            return int(value) if value is not None else None

        def to_float(value):
            return float(value) if value is not None else None
        
        log_data = {
            "run_id": str(run_context.get("run_id", "unknown")),
            "table_name": str(run_context.get("table_name", "unknown")),
            "log_datetime": run_context.get("log_datetime", datetime.now()),
            "status": str(run_context.get("status", "UNKNOWN")),
            "load_type": str(run_context.get("load_type")) if run_context.get("load_type") else None,
            "error_message": str(run_context.get("error_message"))[:1000] if run_context.get("error_message") else None,
            
            # Timing
            "duration_seconds": to_float(run_context.get("duration_seconds")),
            "duration_minutes": to_float(run_context.get("duration_minutes")),
            "read_duration_seconds": to_float(run_context.get("read_duration_seconds")),
            "write_duration_seconds": to_float(run_context.get("write_duration_seconds")),
            
            # Counts
            "source_row_count": to_int(run_context.get("source_row_count")),
            "target_row_count": to_int(run_context.get("target_row_count")),
            "inserted": to_int(run_context.get("inserted")),
            "updated": to_int(run_context.get("updated")),
            "deleted": to_int(run_context.get("deleted")),
            "copied": to_int(run_context.get("copied")),
            
            # Config
            "incremental_interval": to_int(run_context.get("incremental_interval")),
            "was_forced_full_load": bool(run_context.get("was_forced_full_load", False))
        }
        
        log_df = spark.createDataFrame([log_data], schema=log_schema)
        log_df.write.format("delta").mode("append").saveAsTable(full_table_name)
        
        # Enhanced logging
        if log_data["status"] == "SUCCESS":
            duration_text = (
                f"{log_data['duration_minutes']:.2f}min"
                if log_data["duration_minutes"] is not None
                else "duration unknown"
            )

            timing_parts = []
            if log_data["read_duration_seconds"] is not None:
                timing_parts.append(f"R:{log_data['read_duration_seconds']:.1f}s")
            if log_data["write_duration_seconds"] is not None:
                timing_parts.append(f"W:{log_data['write_duration_seconds']:.1f}s")

            timing_text = f" ({' + '.join(timing_parts)})" if timing_parts else ""

            logger.info(
                f"Run logged successfully. "
                f"Source: {log_data['source_row_count'] if log_data['source_row_count'] is not None else 'N/A'} | "
                f"Target: {log_data['target_row_count'] if log_data['target_row_count'] is not None else 'N/A'} | "
                f"Inserted: {log_data['inserted'] if log_data['inserted'] is not None else 'N/A'} | "
                f"Updated: {log_data['updated'] if log_data['updated'] is not None else 'N/A'} | "
                f"Deleted: {log_data['deleted'] if log_data['deleted'] is not None else 'N/A'} | "
                f"{duration_text}{timing_text}"
            )

        elif log_data["status"] == "SKIPPED":
            logger.info("Run logged as skipped.")

        else:
            logger.error(
                f"Run logged as failed. "
                f"Error: {log_data['error_message'][:100] if log_data['error_message'] else 'Unknown'}"
            )

    except Exception as e:
        logger.error(f"Failed to log run to {full_table_name}: {e}")
        logger.error(traceback.format_exc())